In [1]:
import torch
import numpy as np

logit = np.log(0.7 / (1-0.7))
logit_mean = torch.tensor([[logit]])
logit_var = torch.tensor([[0.597224577336219]])

def proproc_variance_input(logit_var):
    return torch.sqrt(logit_var)

In [2]:
num_samples = 25

In [3]:
logit_shape = (logit_mean.shape[0], num_samples, logit_mean.shape[-1])
logit_shape

(1, 25, 1)

In [4]:
logit_var = proproc_variance_input(logit_var)
logit_mean_expanded = logit_mean.unsqueeze(1).expand(-1, num_samples, -1)
logit_var_expanded = logit_var.unsqueeze(1).expand(-1, num_samples, -1)

In [5]:
logit_samples = torch.normal(mean=logit_mean_expanded, std=torch.abs(logit_var_expanded))



In [6]:
prob_samples = torch.sigmoid(logit_samples)
prob_samples

tensor([[[0.7993],
         [0.7816],
         [0.9316],
         [0.3021],
         [0.7810],
         [0.7225],
         [0.8241],
         [0.7546],
         [0.2004],
         [0.6699],
         [0.8128],
         [0.7807],
         [0.8360],
         [0.8933],
         [0.7616],
         [0.7261],
         [0.6830],
         [0.6117],
         [0.7155],
         [0.5938],
         [0.6570],
         [0.3918],
         [0.5677],
         [0.6465],
         [0.6220]]], dtype=torch.float64)

In [10]:


class TwoHeadModel(torch.nn.Module):
    def __init__(self):
        super(TwoHeadModel, self).__init__()
        self.linear = torch.nn.Linear(6, 3)
        self.lrelu = torch.nn.LeakyReLU()
        self.mean_logit_head = torch.nn.Linear(3, 1)
        self.softplus = torch.nn.Softplus()
        self.var_logit_head = torch.nn.Linear(3, 1)
        
    def forward(self, x, untangle_uncertainty = False, N_samples=None):
        x = self.linear(x)
        mean_logit = self.mean_logit_head(self.lrelu(x))
        var_logit = self.var_logit_head(self.softplus(x))

        if not untangle_uncertainty:
            prediction = self.sampling_sigmoind(mean_logit, var_logit)
            return prediction
        else:

            return mean_logit, var_logit
    
    def sampling_sigmoind(mean_logit, var_logit):
        
        num_samples = 10

        #logit_shape = (logit_mean.shape[0], num_samples, logit_mean.shape[-1])

        var_logit = proproc_variance_input(var_logit)
        logit_mean_expanded = logit_mean.unsqueeze(1).expand(-1, num_samples, -1)
        logit_var_expanded = logit_var.unsqueeze(1).expand(-1, num_samples, -1)
        logit_samples = torch.normal(mean=logit_mean_expanded, std=torch.abs(logit_var_expanded))

        prob_samples = torch.sigmoid(logit_samples) # turn logits into probs
        probs = torch.mean(prob_samples, dim=1) # THIS IS USED FOR TRAINING

        return probs


    def proproc_variance_input(logit_var):
        return torch.sqrt(logit_var)
        
    


play_model = TwoHeadModel()

